# FraudGuard - Kaggle Submission
## Notebook 4 - Test Set Prediction

**Objective:** Apply the trained model to the Kaggle test set
and generate a submission file for leaderboard scoring.

**Pipeline:** Load test data -> feature engineering ->
model prediction -> format submission

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded")

Libraries loaded


In [2]:
# Load training data
print("Loading training data...")
df_trans = pd.read_csv('../data/train_transaction.csv')
df_identity = pd.read_csv('../data/train_identity.csv')

df_train = df_trans.merge(df_identity,
                          on='TransactionID',
                          how='left')

print(f"Training data: {df_train.shape}")
print(f"Fraud rate: {df_train['isFraud'].mean()*100:.3f}%")

Loading training data...
Training data: (590540, 434)
Fraud rate: 3.499%


In [3]:
# Load test data
print("Loading test data...")
df_test_trans = pd.read_csv('../data/test_transaction.csv')
df_test_id = pd.read_csv('../data/test_identity.csv')

df_test = df_test_trans.merge(df_test_id,
                              on='TransactionID',
                              how='left')

# Save TransactionID for submission
test_ids = df_test['TransactionID'].copy()

print(f"Test data: {df_test.shape}")
print(f"Test TransactionIDs: {len(test_ids):,}")

Loading test data...
Test data: (506691, 433)
Test TransactionIDs: 506,691


In [4]:
def engineer_features(df):
    """Apply full feature engineering pipeline to any dataset."""

    # Missing data strategy
    missing_pct = (df.isnull().sum(axis=0) / len(df) * 100)
    tier1_drop = missing_pct[missing_pct > 99].index.tolist()
    tier2_indicator = missing_pct[
        (missing_pct >= 50) & (missing_pct <= 99)
    ].index.tolist()
    tier3_impute = missing_pct[
        (missing_pct > 0) & (missing_pct < 50)
    ].index.tolist()

    # Drop Tier 1
    df = df.drop(columns=tier1_drop, errors='ignore')

    # Binary indicators for Tier 2
    for col in tier2_indicator:
        if col in df.columns:
            df[f'{col}_missing'] = df[col].isnull().astype(int)

    # Impute
    numeric_missing = [c for c in tier2_indicator + tier3_impute
                       if c in df.columns and
                       df[c].dtype != 'object']
    for col in numeric_missing:
        df[col] = df[col].fillna(df[col].median())

    cat_missing = [c for c in tier2_indicator + tier3_impute
                   if c in df.columns and
                   df[c].dtype == 'object']
    for col in cat_missing:
        df[col] = df[col].fillna(df[col].mode()[0])

    # Time features
    df['tx_hour'] = (df['TransactionDT'] // 3600) % 24
    df['tx_day'] = (df['TransactionDT'] // (3600 * 24)) % 7
    df['tx_day_elapsed'] = df['TransactionDT'] // (3600 * 24)
    df['is_night'] = (
        (df['tx_hour'] >= 23) | (df['tx_hour'] <= 6)
    ).astype(int)
    df['is_weekend'] = (df['tx_day'] >= 5).astype(int)

    # Fix zero income
    if 'annual_inc' not in df.columns:
        df['annual_inc'] = 0

    # Velocity features
    df = df.sort_values(['card1', 'TransactionDT']).reset_index(
        drop=True)
    df['card_tx_count'] = df.groupby('card1')[
        'TransactionID'].transform('count')
    df['card_mean_amt'] = df.groupby('card1')[
        'TransactionAmt'].transform('mean')
    df['amt_deviation'] = (
        df['TransactionAmt'] - df['card_mean_amt']
    ) / (df['card_mean_amt'] + 1)
    df['amt_ratio'] = df['TransactionAmt'] / (
        df['card_mean_amt'] + 1)

    # Categorical encoding
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()

    for col in cat_cols:
        n_unique = df[col].nunique()
        if n_unique < 10:
            df[col] = le.fit_transform(df[col].astype(str))
        else:
            freq_map = df[col].value_counts().to_dict()
            df[col] = df[col].map(freq_map)

    return df

print("Feature engineering function defined")

Feature engineering function defined


In [5]:
def engineer_features(df):
    """Apply full feature engineering pipeline to any dataset."""

    # Missing data strategy
    missing_pct = (df.isnull().sum(axis=0) / len(df) * 100)
    tier1_drop = missing_pct[missing_pct > 99].index.tolist()
    tier2_indicator = missing_pct[
        (missing_pct >= 50) & (missing_pct <= 99)
    ].index.tolist()
    tier3_impute = missing_pct[
        (missing_pct > 0) & (missing_pct < 50)
    ].index.tolist()

    # Drop Tier 1
    df = df.drop(columns=tier1_drop, errors='ignore')

    # Binary indicators for Tier 2
    for col in tier2_indicator:
        if col in df.columns:
            df[f'{col}_missing'] = df[col].isnull().astype(int)

    # Impute
    numeric_missing = [c for c in tier2_indicator + tier3_impute
                       if c in df.columns and
                       df[c].dtype != 'object']
    for col in numeric_missing:
        df[col] = df[col].fillna(df[col].median())

    cat_missing = [c for c in tier2_indicator + tier3_impute
                   if c in df.columns and
                   df[c].dtype == 'object']
    for col in cat_missing:
        df[col] = df[col].fillna(df[col].mode()[0])

    # Time features
    df['tx_hour'] = (df['TransactionDT'] // 3600) % 24
    df['tx_day'] = (df['TransactionDT'] // (3600 * 24)) % 7
    df['tx_day_elapsed'] = df['TransactionDT'] // (3600 * 24)
    df['is_night'] = (
        (df['tx_hour'] >= 23) | (df['tx_hour'] <= 6)
    ).astype(int)
    df['is_weekend'] = (df['tx_day'] >= 5).astype(int)

    # Fix zero income
    if 'annual_inc' not in df.columns:
        df['annual_inc'] = 0

    # Velocity features
    df = df.sort_values(['card1', 'TransactionDT']).reset_index(
        drop=True)
    df['card_tx_count'] = df.groupby('card1')[
        'TransactionID'].transform('count')
    df['card_mean_amt'] = df.groupby('card1')[
        'TransactionAmt'].transform('mean')
    df['amt_deviation'] = (
        df['TransactionAmt'] - df['card_mean_amt']
    ) / (df['card_mean_amt'] + 1)
    df['amt_ratio'] = df['TransactionAmt'] / (
        df['card_mean_amt'] + 1)

    # Categorical encoding
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()

    for col in cat_cols:
        n_unique = df[col].nunique()
        if n_unique < 10:
            df[col] = le.fit_transform(df[col].astype(str))
        else:
            freq_map = df[col].value_counts().to_dict()
            df[col] = df[col].map(freq_map)

    return df

print("Feature engineering function defined")

Feature engineering function defined


In [7]:
# Define feature columns
feature_cols = [
    'TransactionAmt', 'ProductCD', 'card1', 'card2',
    'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2',
    'tx_hour', 'tx_day', 'tx_day_elapsed', 'is_night',
    'is_weekend', 'card_tx_count', 'card_mean_amt',
    'amt_deviation', 'amt_ratio', 'P_emaildomain',
    'R_emaildomain', 'dist1', 'C1', 'C2', 'C4', 'C5',
    'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13',
    'C14', 'D1', 'D2', 'D3', 'D4', 'D10', 'D15',
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6',
    'id_01', 'id_02', 'id_03', 'id_05', 'id_06',
    'id_09', 'id_10', 'id_11', 'id_13', 'id_17',
    'id_19', 'id_20', 'DeviceType'
]

print("Engineering training features...")
df_train_eng = engineer_features(df_train)

print("Engineering test features...")
df_test_eng = engineer_features(df_test)

# Keep only feature columns that exist in both
common_features = [f for f in feature_cols
                   if f in df_train_eng.columns
                   and f in df_test_eng.columns]

print(f"\nCommon features: {len(common_features)}")

X_train = df_train_eng[common_features]
y_train = df_train_eng['isFraud']
X_test_final = df_test_eng[common_features]

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test_final.shape}")

Engineering training features...
Engineering test features...

Common features: 48
Training set: (590540, 48)
Test set:     (506691, 48)


In [8]:
# SMOTE
print("Applying SMOTE...")
smote = SMOTE(sampling_strategy=0.1,
              random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"After SMOTE: {X_train_sm.shape}")

# Train on full training data
print("\nTraining model on full dataset...")
model = XGBClassifier(
    n_estimators=300, max_depth=6,
    learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.1,
    reg_lambda=1.0, random_state=42,
    eval_metric='aucpr', verbosity=0
)
model.fit(X_train_sm, y_train_sm)
print("Model trained successfully")

# Generate predictions
print("\nGenerating test predictions...")
test_preds = model.predict_proba(X_test_final)[:, 1]

print(f"Predictions generated: {len(test_preds):,}")
print(f"Mean predicted fraud probability: "
      f"{test_preds.mean()*100:.3f}%")

Applying SMOTE...
After SMOTE: (626864, 48)

Training model on full dataset...
Model trained successfully

Generating test predictions...
Predictions generated: 506,691
Mean predicted fraud probability: 6.135%


In [9]:
# SMOTE
print("Applying SMOTE...")
smote = SMOTE(sampling_strategy=0.1,
              random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"After SMOTE: {X_train_sm.shape}")

# Train on full training data
print("\nTraining model on full dataset...")
model = XGBClassifier(
    n_estimators=300, max_depth=6,
    learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.1,
    reg_lambda=1.0, random_state=42,
    eval_metric='aucpr', verbosity=0
)
model.fit(X_train_sm, y_train_sm)
print("Model trained successfully")

# Generate predictions
print("\nGenerating test predictions...")
test_preds = model.predict_proba(X_test_final)[:, 1]

print(f"Predictions generated: {len(test_preds):,}")
print(f"Mean predicted fraud probability: "
      f"{test_preds.mean()*100:.3f}%")

Applying SMOTE...
After SMOTE: (626864, 48)

Training model on full dataset...
Model trained successfully

Generating test predictions...
Predictions generated: 506,691
Mean predicted fraud probability: 6.135%


In [10]:
# Format submission
submission = pd.DataFrame({
    'TransactionID': test_ids,
    'isFraud': test_preds
})

# Save
submission.to_csv('../data/submission.csv', index=False)

print(f"Submission saved: {submission.shape}")
print(f"\nSample:")
print(submission.head(10))

Submission saved: (506691, 2)

Sample:
   TransactionID   isFraud
0        3663549  0.010988
1        3663550  0.061100
2        3663551  0.165158
3        3663552  0.008297
4        3663553  0.029594
5        3663554  0.037244
6        3663555  0.199223
7        3663556  0.013427
8        3663557  0.035037
9        3663558  0.024039
